# AI-driven Autonomous Kitting System

**Physical AI 応用1｜再現実験Notebook**  
**実証内容**：QR画像の読取 → SOP取得 → Qwen2.5-0.5B-Instructによる再推論 → 独立したPython Validator → AGV模擬搬送 → Franka Pandaによる3工程キッティング。

このNotebookは、2026年10月29日にGoogle Colabで製品A/Bの統合実行が**それぞれ1回成功したv2**から作成しました。**ロボットの制御アルゴリズムと実行処理は変更せず**、セル出力・実行履歴のみ除去しています。新しいランタイムでの再実行結果は環境依存のため保証しません。

### 実行方法
1. Google Colab（Python 3）で開き、最初の `%pip` セルから **ランタイム → すべてのセルを実行**。
2. Google Driveへのマウントを許可。Driveを使わない場合は設定セルの `SAVE_TO_DRIVE = False` に変更。
3. `ALL PRODUCT CAPTURES SUCCESS: True` を確認する。処理は製品A→製品Bの順番に実施する。
4. 生成物は `PAI2026_Submission_Artifacts/run_YYYYMMDDTHHMMSSZ/` 以下に製品別に保存されます。

**動作確認済み環境（実験ログ）**：Google Colabクラウド／Python 3.13.18／CPU／PyBullet 3.2.7／NumPy 2.1.3／PyTorch 2.11.0+cpu／Transformers 4.57.6。`%pip` セルは対応範囲を指定しており、全依存ライブラリを同一バージョンに固定するロックではありません。

**機能の境界**：QRはPNGをOpenCVで読取（PyBullet内カメラではない）。AGVは車輪制御のない運動学的模擬搬送で、視覚専用の荷台を使用。SOP照合は所定の定型文に限定。部品色と座標は既知（VLMによる認識・VLA制御ではない）。

記録済みの実験データとレポートはGitHubリポジトリの `assets/` と `reports/` に収録しています。


## 1｜依存関係と実行条件

In [ ]:
%pip -q install pybullet "transformers>=4.45,<5" accelerate pillow "qrcode[pil]" imageio imageio-ffmpeg


In [ ]:
import json, re, time, unicodedata
import numpy as np
import torch
import pybullet as p
import pybullet_data
from PIL import Image
from IPython.display import display
from transformers import AutoTokenizer, AutoModelForCausalLM
from importlib.metadata import version

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
SARASHINA_ID = "sbintuitions/sarashina2.2-0.5b-instruct-v0.1"
RUN_MODEL_COMPARISON = False  # 比較評価を再実行するときだけTrueへ

SOP_TEXT = """作業手順書 SOP-001
目的：棚にある部品を指定されたトレーへ仕分ける。
工程1：赤色の部品をトレーAの区画1に配置する。
工程2：青色の部品をトレーBの区画1に配置する。
工程3：緑色の部品をトレーAの区画2に配置する。
注意事項：
- 工程1から順番に実行する。
- 指定部品がない場合、別の部品で代用しない。
- 配置先が占有されている場合、作業を停止する。
- 不明確な指示は推測せず確認する。
"""
print("Versions:", {k: version(k) for k in ["pybullet", "transformers", "torch", "numpy"]})
print("SOP:", SOP_TEXT)


# Run controls. Keep actual model inference on both products.
PRODUCT_SEQUENCE = ["PRODUCT_A", "PRODUCT_B"]
CAPTURE_ROBOT_VIDEO = True
VIDEO_W, VIDEO_H, VIDEO_FPS = 640, 400, 8
VIDEO_CAPTURE_EVERY_STEPS = 240  # 1 in 240 physics simulation steps
# All run artifacts will be saved to the user's Drive, in unique run folders.
SAVE_TO_DRIVE = True


## 2｜元の成功版に基づくPyBullet環境初期化（関数化のみ）

In [ ]:
def initialize_scene():
    global plane, robot, ee_link, home_joints, object_positions, slot_positions, object_colors, objects, light_markers, light_status, indicator_positions, set_light, observe_slot_occupancy, view, projection
    # ======================================================
    # STEP 10A: INDUSTRIAL SORTING SCENE
    # Shelf + Trays + Slot Indicators
    # No robot manipulation yet
    # ======================================================

    import pybullet as p
    import pybullet_data
    import numpy as np
    from PIL import Image
    from IPython.display import display

    # 1. Reset the simulation to a clean initial state
    if p.isConnected():
        p.disconnect()
    p.connect(p.DIRECT)
    p.resetSimulation()
    p.setAdditionalSearchPath(pybullet_data.getDataPath())
    p.setGravity(0, 0, -9.81)
    p.setTimeStep(1 / 240)

    plane = p.loadURDF("plane.urdf")
    robot = p.loadURDF(
        "franka_panda/panda.urdf",
        useFixedBase=True
    )
    ee_link = 11

    home_joints = [
        0.0, -0.785, 0.0,
        -2.356, 0.0, 1.571, 0.785
    ]

    for j, angle in enumerate(home_joints):
        p.resetJointState(robot, j, angle)
        p.setJointMotorControl2(
            robot, j, p.POSITION_CONTROL,
            targetPosition=angle, force=87
        )

    for j in [9, 10]:
        p.resetJointState(robot, j, 0.04)
        p.setJointMotorControl2(
            robot, j, p.POSITION_CONTROL,
            targetPosition=0.04, force=20
        )

    # 2. Part and destination coordinates
    # Reuse the previously validated picking positions
    object_positions = {
        "red":   [0.55,  0.16, 0.025],
        "blue":  [0.55,  0.00, 0.025],
        "green": [0.55, -0.16, 0.025]
    }

    # Separate slots to prevent parts from overlapping
    slot_positions = {
        "A1": [0.43,  0.16],
        "A2": [0.43,  0.33],
        "B1": [0.43, -0.20]
    }

    object_colors = {
        "red":   [0.95, 0.15, 0.15, 1],
        "blue":  [0.10, 0.30, 1.00, 1],
        "green": [0.10, 0.75, 0.30, 1]
    }

    # 3. Helper: create visual-only panels
    def add_panel(position, half_extents, color):
        visual = p.createVisualShape(
            p.GEOM_BOX,
            halfExtents=half_extents,
            rgbaColor=color
        )
        return p.createMultiBody(
            baseMass=0,
            baseVisualShapeIndex=visual,
            basePosition=position
        )

    # Shelf footprint
    add_panel(
        [0.585, 0.00, 0.001],
        [0.065, 0.235, 0.002],
        [0.55, 0.61, 0.69, 0.65]
    )

    # Tray A (two compartments)
    add_panel(
        [0.43, 0.245, 0.001],
        [0.073, 0.166, 0.002],
        [0.25, 0.75, 0.47, 0.55]
    )

    # Tray B (one compartment)
    add_panel(
        [0.43, -0.20, 0.001],
        [0.073, 0.075, 0.002],
        [0.32, 0.50, 0.95, 0.55]
    )

    # Visual compartment markers
    for slot, xy in slot_positions.items():
        add_panel(
            [xy[0], xy[1], 0.004],
            [0.058, 0.052, 0.001],
            [0.95, 0.95, 0.95, 0.55]
        )

    # 4. Create physical parts
    objects = {}

    for color, pos in object_positions.items():
        collision = p.createCollisionShape(
            p.GEOM_BOX,
            halfExtents=[0.025, 0.025, 0.025]
        )

        visual = p.createVisualShape(
            p.GEOM_BOX,
            halfExtents=[0.025, 0.025, 0.025],
            rgbaColor=object_colors[color]
        )

        body = p.createMultiBody(
            baseMass=0.05,
            baseCollisionShapeIndex=collision,
            baseVisualShapeIndex=visual,
            basePosition=pos
        )

        p.changeDynamics(
            body, -1, lateralFriction=2.0
        )

        objects[color] = body

    # 5. Indicator lights: simulator-state information
    light_markers = {}
    light_status = {}

    indicator_positions = {
        "red":   [0.64,  0.16, 0.02],
        "blue":  [0.64,  0.00, 0.02],
        "green": [0.64, -0.16, 0.02],
        "A1":    [0.36,  0.16, 0.02],
        "A2":    [0.36,  0.33, 0.02],
        "B1":    [0.36, -0.20, 0.02]
    }

    for name, pos in indicator_positions.items():
        visual = p.createVisualShape(
            p.GEOM_SPHERE,
            radius=0.012,
            rgbaColor=[0.25, 0.25, 0.25, 1]
        )

        body = p.createMultiBody(
            baseMass=0,
            baseVisualShapeIndex=visual,
            basePosition=pos
        )

        light_markers[name] = body
        light_status[name] = False

    def set_light(name, enabled):
        if name not in light_markers:
            raise ValueError(f"Unknown indicator: {name}")

        light_status[name] = bool(enabled)

        color = (
            [1.0, 0.95, 0.05, 1]
            if enabled
            else [0.25, 0.25, 0.25, 1]
        )

        p.changeVisualShape(
            light_markers[name], -1,
            rgbaColor=color
        )

    # Illuminate Step 1 of SOP-001
    set_light("red", True)
    set_light("A1", True)

    # 6. Occupancy observation from physics state
    def observe_slot_occupancy():
        occupied = {}

        for slot, xy in slot_positions.items():
            occupants = []

            for name, body in objects.items():
                pos = np.array(
                    p.getBasePositionAndOrientation(body)[0]
                )

                horizontal_distance = np.linalg.norm(
                    pos[:2] - np.array(xy)
                )

                # Count parts resting within the slot
                if horizontal_distance < 0.045 and pos[2] < 0.06:
                    occupants.append(name)

            occupied[slot] = occupants

        return occupied

    # 7. Stabilize
    for _ in range(240):
        p.stepSimulation()

    # 8. Validation
    scene_ok = True

    print("=== STEP 10A SCENE VALIDATION ===")

    for name, body in objects.items():
        actual = np.array(
            p.getBasePositionAndOrientation(body)[0]
        )
        error = np.linalg.norm(
            actual - np.array(object_positions[name])
        )
        valid = error < 0.01
        scene_ok = scene_ok and valid

        print(
            name,
            "position =", np.round(actual, 4),
            "valid =", valid
        )

    occupancy = observe_slot_occupancy()

    print("\nSlot occupancy:", occupancy)
    print("Lights ON:", [
        k for k, v in light_status.items() if v
    ])

    slots_empty = all(
        len(parts) == 0
        for parts in occupancy.values()
    )

    scene_ok = scene_ok and slots_empty

    print("All slots empty:", slots_empty)
    print("Scene ready:", scene_ok)
    assert scene_ok, "STOP: Scene initialization failed"

    # 9. Render
    view = p.computeViewMatrixFromYawPitchRoll(
        cameraTargetPosition=[0.47, 0.07, 0.18],
        distance=1.65,
        yaw=55,
        pitch=-45,
        roll=0,
        upAxisIndex=2
    )

    projection = p.computeProjectionMatrixFOV(
        fov=60,
        aspect=1.5,
        nearVal=0.1,
        farVal=5
    )

    w, h, rgba, _, _ = p.getCameraImage(
        960, 640,
        viewMatrix=view,
        projectionMatrix=projection,
        renderer=p.ER_TINY_RENDERER
    )

    display(
        Image.fromarray(
            np.asarray(rgba, dtype=np.uint8).reshape(h, w, 4)
        ).convert("RGB")
    )


## 3｜既存のFranka Panda IK・把持制御（アルゴリズムは変更しない）

In [ ]:
# 基本制御のパラメータ（成功版Step 6から移植）
lower_limits = [-2.8973, -1.7628, -2.8973, -3.0718, -2.8973, -0.0175, -2.8973]
upper_limits = [ 2.8973,  1.7628,  2.8973, -0.0698,  2.8973,  3.7525,  2.8973]
joint_ranges = [u - l for l, u in zip(lower_limits, upper_limits)]
orientation = p.getQuaternionFromEuler([np.pi, 0, 0])

# -----------------------------------------------------

def ee_position():
    return np.array(
        p.getLinkState(
            robot, ee_link,
            computeForwardKinematics=True
        )[4]
    )

def cube_position():
    return np.array(
        p.getBasePositionAndOrientation(cube)[0]
    )

def finger_contacts():
    contacts = p.getContactPoints(
        bodyA=robot, bodyB=cube
    )
    return {c[3] for c in contacts}

def control_fingers(open_gripper):
    target = 0.04 if open_gripper else 0.0

    for j in [9, 10]:
        p.setJointMotorControl2(
            robot, j,
            p.POSITION_CONTROL,
            targetPosition=target,
            force=20,
            maxVelocity=0.08
        )

def move_to(target, steps=960, velocity=0.12,
            gripper_open=True):

    target = np.array(target, dtype=float)

    current = [
        p.getJointState(robot, j)[0]
        for j in range(7)
    ]

    solution = p.calculateInverseKinematics(
        robot,
        ee_link,
        target.tolist(),
        orientation,
        lowerLimits=lower_limits,
        upperLimits=upper_limits,
        jointRanges=joint_ranges,
        restPoses=current,
        maxNumIterations=500,
        residualThreshold=1e-5
    )

    for _ in range(steps):
        for j in range(7):
            p.setJointMotorControl2(
                robot, j,
                p.POSITION_CONTROL,
                targetPosition=solution[j],
                force=87,
                maxVelocity=velocity
            )

        control_fingers(gripper_open)
        p.stepSimulation()

    error = np.linalg.norm(
        ee_position() - target
    )

    return float(error)

def hold_and_move_fingers(open_gripper, steps=480):
    # Hold arm at its present joint angles
    arm_hold = [
        p.getJointState(robot, j)[0]
        for j in range(7)
    ]

    for _ in range(steps):
        for j in range(7):
            p.setJointMotorControl2(
                robot, j,
                p.POSITION_CONTROL,
                targetPosition=arm_hold[j],
                force=87
            )

        control_fingers(open_gripper)
        p.stepSimulation()

def require(condition, message):
    if not condition:
        raise RuntimeError(message)


## 4｜SOPの定型照合と共有Preflight Gate

In [ ]:
COLOR_MAP = {
    "赤": "red", "赤色": "red", "red": "red",
    "青": "blue", "青色": "blue", "blue": "blue",
    "緑": "green", "緑色": "green", "green": "green",
}
ALLOWED_SLOTS = {"A1", "A2", "B1"}

def normalize_color(value):
    if not isinstance(value, str):
        return None
    return COLOR_MAP.get(value.strip())

def normalize_slot(value):
    if not isinstance(value, str):
        return None
    value = unicodedata.normalize("NFKC", value).strip()
    if value in ALLOWED_SLOTS:
        return value
    match = re.fullmatch(r"トレー\s*([AB])\s*の?\s*区画\s*([12])", value)
    if match:
        slot = match.group(1) + match.group(2)
        return slot if slot in ALLOWED_SLOTS else None
    return None

def verify_against_sop(content, color, slot):
    # このフォーマット以外は明示的に拒否する（汎用日本語理解とは主張しない）
    pattern = (
        r"^\s*(赤色?|青色?|緑色?)の部品を"
        r"トレー([AB])の区画([12])に配置する。\s*$"
    )
    match = re.fullmatch(pattern, content)
    if not match:
        return False
    expected_color = COLOR_MAP[match.group(1)]
    expected_slot = match.group(2) + match.group(3)
    return color == expected_color and slot == expected_slot

def parse_json_response(response):
    raw = response.strip()
    fenced = re.fullmatch(
        r"```(?:json)?[ \t]*\r?\n(.*?)\r?\n```[ \t]*",
        raw, flags=re.DOTALL | re.IGNORECASE
    )
    if fenced:
        raw = fenced.group(1).strip()
    return json.loads(raw)

sop_lines = re.findall(
    r"^工程\s*(\d+)\s*[：:]\s*(.+)$",
    SOP_TEXT, flags=re.MULTILINE
)
if not (len(sop_lines) == 3 and [int(n) for n, _ in sop_lines] == [1, 2, 3]):
    raise RuntimeError("STOP: Invalid SOP step count or order")

# 定型照合器の事前自己テスト
assert verify_against_sop("赤色の部品をトレーAの区画1に配置する。", "red", "A1")
assert not verify_against_sop("赤色の部品をトレーAの区画1に配置する。", "green", "A1")
assert normalize_slot("トレーＡの区画２") == "A2"
print("Independent SOP validator: READY")


def part_position(color):
    return np.array(p.getBasePositionAndOrientation(objects[color])[0])

def validate_preflight(plan):
    occupied = observe_slot_occupancy()
    for task in plan:
        color, slot = task["object_color"], task["slot"]
        if color not in objects:
            raise RuntimeError(f"Missing part: {color}")
        if slot not in slot_positions:
            raise RuntimeError(f"Unknown destination: {slot}")
        if np.linalg.norm(part_position(color) - np.array(object_positions[color])) >= 0.04:
            raise RuntimeError(f"Missing part at source: {color}")
        if occupied[slot]:
            raise RuntimeError(f"Occupied slot: {slot}")
    return True


## 5｜Qwenモデルを一度だけロード

In [ ]:
print("Loading Qwen in Colab runtime (no inference API)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
llm = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=dtype
).to(device)
llm.eval()
print("Model:", MODEL_ID, "| Device:", device)


## 6｜製品A/BのSOP、Qwen計画生成、QR読み取り

In [ ]:
PRODUCT_SOPS = {
    "PRODUCT_A": SOP_TEXT,

    "PRODUCT_B": """作業手順書 SOP-002
目的：製品Bの生産準備に必要な部品を仕分ける。
工程1：赤色の部品をトレーAの区画1に配置する。
工程2：青色の部品をトレーAの区画2に配置する。
工程3：緑色の部品をトレーBの区画1に配置する。
注意事項：
- 工程1から順番に実行する。
- 指定部品がない場合、別の部品で代用しない。
- 配置先が占有されている場合、作業を停止する。
- 不明確な指示は推測せず確認する。
"""
}
planning_prompt = """
あなたは製造現場の作業手順解析AIです。
入力された1工程から、部品の色と配置先を抽出します。

色の対応：
赤・赤色 = red
青・青色 = blue
緑・緑色 = green

配置先の例：
トレーAの区画1 = A1
トレーAの区画2 = A2
トレーBの区画1 = B1

出力形式：
{"object_color":"red","slot":"A1"}

ルール：
- 入力に書かれた情報のみを抽出する
- 部品と配置先の対応を変更しない
- 不明な情報を推測しない
- JSONのみを返す
"""
def generate_product_plan(recipe_id, sop):

    lines = re.findall(
        r"^工程\s*(\d+)\s*[：:]\s*(.+)$",
        sop, flags=re.MULTILINE
    )

    if len(lines) != 3 or [
        int(n) for n, _ in lines
    ] != [1, 2, 3]:
        raise RuntimeError("STOP: Invalid SOP structure")

    plan = []
    raw_records = []

    for number, instruction in lines:

        messages = [
            {"role": "system", "content": planning_prompt},
            {"role": "user", "content": instruction}
        ]

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = tokenizer(
            prompt, return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            output = llm.generate(
                **inputs,
                max_new_tokens=100,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        response = tokenizer.decode(
            output[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True
        ).strip()

        raw_records.append({
            "step": int(number),
            "instruction": instruction,
            "raw_output": response
        })

        # Validate LLM output
        try:
            parsed = parse_json_response(response)

            if (
                not isinstance(parsed, dict)
                or set(parsed) != {"object_color", "slot"}
            ):
                raise ValueError("Invalid JSON schema")

            color = normalize_color(parsed["object_color"])
            slot = normalize_slot(parsed["slot"])

            if (
                color is None
                or slot is None
                or not verify_against_sop(
                    instruction, color, slot
                )
            ):
                raise ValueError("SOP mismatch")

        except (ValueError, TypeError, KeyError) as exc:
            raise RuntimeError(
                f"STOP: {recipe_id} step {number}: {exc}"
            ) from exc

        plan.append({
            "step": int(number),
            "object_color": color,
            "slot": slot
        })

        print(f"Step {number}: {color} -> {slot} | PASS")

    # Check quantity, duplicates, order and slots
    colors = [x["object_color"] for x in plan]
    slots = [x["slot"] for x in plan]

    if (
        len(plan) != 3
        or len(set(colors)) != 3
        or len(set(slots)) != 3
        or set(colors) != {"red", "blue", "green"}
        or set(slots) != {"A1", "A2", "B1"}
    ):
        raise RuntimeError("STOP: Invalid part/slot allocation")

    return {
        "recipe_id": recipe_id,
        "sop": sop,
        "plan": plan,
        "raw_records": raw_records,
        "validated": True
    }

import cv2, qrcode
from pathlib import Path

WORK_ORDERS = {"WO-2026-A": "PRODUCT_A", "WO-2026-B": "PRODUCT_B"}
PRODUCT_QRS = {product: wo for wo, product in WORK_ORDERS.items()}

def scan_qr_and_plan(product_id, output_dir):
    """QR image -> real OpenCV decode -> SOP -> fresh Qwen inference -> independent gate."""
    if product_id not in PRODUCT_QRS:
        raise RuntimeError("STOP: unknown product")
    work_order_id = PRODUCT_QRS[product_id]
    qr_path = output_dir / f"qr_{product_id}.png"
    qrcode.make(work_order_id).save(qr_path)
    image = cv2.imread(str(qr_path))
    assert image is not None, "QR file missing"
    decoded_id, pts, _ = cv2.QRCodeDetector().detectAndDecode(image)
    if decoded_id != work_order_id or pts is None:
        raise RuntimeError("STOP: QR decode did not match work order")
    if decoded_id not in WORK_ORDERS:
        raise RuntimeError("STOP: unknown work order")
    resolved_product = WORK_ORDERS[decoded_id]
    sop = PRODUCT_SOPS[resolved_product]
    print(f"QR: {decoded_id} -> {resolved_product}")
    t0 = time.perf_counter()
    result = generate_product_plan(resolved_product, sop)  # actual Qwen inference
    inference_s = time.perf_counter() - t0
    if not result["validated"]:
        raise RuntimeError("STOP: model plan not validated")
    steps = re.findall(r"^工程\s*(\d+)\s*[：:]\s*(.+)$", sop, flags=re.MULTILINE)
    plan = result["plan"]
    if len(steps) != len(plan) or not plan:
        raise RuntimeError("STOP: SOP step count mismatch")
    if not all(task["step"] == int(n) and verify_against_sop(instr,task["object_color"],task["slot"])
               for task,(n,instr) in zip(plan,steps)):
        raise RuntimeError("STOP: independent SOP validation failed")
    if len({t["slot"] for t in plan}) != len(plan):
        raise RuntimeError("STOP: duplicate destination")
    return {
        "work_order_id":decoded_id, "product_id":resolved_product,
        "qr_image":qr_path.name, "sop":sop,"plan":plan,
        "raw_llm_outputs":result["raw_records"],
        "llm_inference_seconds":float(inference_s)
    }


## 7｜AGV直接ドッキング模擬（荷下ろしの瞬間移動なし）

In [ ]:
def dock_agv(product_id, output_dir):

    # ==========================================================
    # STEP 17B-R: DIRECT PICKING FROM DOCKED AGV
    # AGV transports actual part bodies to original pick locations.
    # Visual-only low-profile cargo platform.
    # No robot movement / no invisible unloading.
    # ==========================================================

    import numpy as np
    import pybullet as p
    from PIL import Image
    from pathlib import Path
    from IPython.display import display, Image as DisplayImage

    print("=== STEP 17B-R: DIRECT AGV PICKING ===")

    COLORS = ["red", "blue", "green"]

    assert p.isConnected()
    assert all(c in objects for c in COLORS)


    # Verify clean robot scene
    original_joints = np.array([
        p.getJointState(robot, j)[0] for j in range(7)
    ])

    for color in COLORS:
        actual = np.array(
            p.getBasePositionAndOrientation(objects[color])[0]
        )
        expected = np.array(object_positions[color])
        assert np.linalg.norm(actual - expected) < 0.01, (
            f"STOP: Reset Step 10A before running: {color}"
        )

    # Vehicle coordinates.
    # At docking, cargo is positioned exactly at existing
    # Panda pickup coordinates: x=0.55, y=+0.16/0/-0.16.
    START_X = 1.30
    DOCK_X = 0.55

    visual_parts = []

    def visual_component(offset, half, rgba):
        shape = p.createVisualShape(
            p.GEOM_BOX,
            halfExtents=half,
            rgbaColor=rgba
        )
        body = p.createMultiBody(
            baseMass=0,
            baseVisualShapeIndex=shape,
            basePosition=[0, 0, -2]
        )
        visual_parts.append((body, np.array(offset)))
        return body

    # Very low cargo platform; not a collision support.
    visual_component(
        [0, 0, 0.004],
        [0.09, 0.22, 0.003],
        [0.70, 0.78, 0.86, 1]
    )

    # Two low side rails outside the three pickup locations
    for y in [-0.225, 0.225]:
        visual_component(
            [0, y, 0.014],
            [0.095, 0.006, 0.010],
            [0.34, 0.44, 0.56, 1]
        )

    # Rear tractor chassis: outside the pickup area
    visual_component(
        [0.30, 0, 0.075],
        [0.115, 0.17, 0.045],
        [0.27, 0.35, 0.45, 1]
    )

    # Connecting tow bar
    visual_component(
        [0.17, 0, 0.045],
        [0.12, 0.024, 0.018],
        [0.44, 0.52, 0.60, 1]
    )

    # Decorative wheels, no wheel-drive simulation
    wheel_shape = p.createVisualShape(
        p.GEOM_CYLINDER,
        radius=0.045,
        length=0.022,
        rgbaColor=[0.11, 0.14, 0.19, 1],
        visualFrameOrientation=p.getQuaternionFromEuler(
            [np.pi/2, 0, 0]
        )
    )

    for dx in [0.22, 0.38]:
        for dy in [-0.18, 0.18]:
            body = p.createMultiBody(
                baseMass=0,
                baseVisualShapeIndex=wheel_shape,
                basePosition=[0, 0, -2]
            )
            visual_parts.append((
                body, np.array([dx, dy, 0.045])
            ))

    def set_vehicle_and_cargo(x):
        for body, offset in visual_parts:
            p.resetBasePositionAndOrientation(
                body,
                (np.array([x, 0, 0]) + offset).tolist(),
                [0, 0, 0, 1]
            )

        for color in COLORS:
            target = [
                float(x),
                float(object_positions[color][1]),
                float(object_positions[color][2])
            ]

            p.resetBasePositionAndOrientation(
                objects[color], target, [0, 0, 0, 1]
            )

            p.resetBaseVelocity(
                objects[color],
                [0, 0, 0],
                [0, 0, 0]
            )

    view_agv = p.computeViewMatrixFromYawPitchRoll(
        cameraTargetPosition=[0.83, 0, 0.12],
        distance=2.25,
        yaw=55,
        pitch=-48,
        roll=0,
        upAxisIndex=2
    )

    proj_agv = p.computeProjectionMatrixFOV(
        fov=60, aspect=1.6,
        nearVal=0.1, farVal=5
    )

    frames = []
    AGV_DIRECT_PICK_READY = False

    try:
        # Start with parts already loaded on the vehicle.
        # No unloading sequence is necessary.
        for fraction in np.linspace(0, 1, 22):
            x = START_X + (
                DOCK_X - START_X
            ) * float(fraction)

            set_vehicle_and_cargo(x)

            w, h, rgba, _, _ = p.getCameraImage(
                640, 400,
                viewMatrix=view_agv,
                projectionMatrix=proj_agv,
                renderer=p.ER_TINY_RENDERER
            )

            frames.append(
                Image.fromarray(
                    np.asarray(rgba, dtype=np.uint8).reshape(
                        h, w, 4
                    )
                ).convert("RGB")
            )

        # Vehicle stays docked; the actual parts stay
        # on the cargo footprint at the Panda pick coordinates.
        for color in COLORS:
            actual = np.array(
                p.getBasePositionAndOrientation(
                    objects[color]
                )[0]
            )
            expected = np.array(object_positions[color])
            error = np.linalg.norm(actual - expected)

            print(
                f"Docked part [{color}]: "
                f"{error*1000:.3f} mm"
            )
            assert error < 0.01

        after_joints = np.array([
            p.getJointState(robot, j)[0] for j in range(7)
        ])

        assert np.allclose(
            original_joints, after_joints, atol=1e-5
        ), "STOP: Robot state changed"

        assert all(np.linalg.norm(np.array(p.getBasePositionAndOrientation(objects[c])[0]) - np.array(object_positions[c])) < 0.01 for c in COLORS)
        assert all(
            not occupants
            for occupants in observe_slot_occupancy().values()
        )

        animation_path = output_dir / f"agv_docking_{product_id}.gif"

        frames[0].save(
            animation_path,
            save_all=True,
            append_images=frames[1:],
            duration=130,
            loop=0
        )

        AGV_DIRECT_PICK_READY = True

        print("\nSTEP 17B-R FINAL RESULT")
        print("AGV docking: PASS")
        print("Actual parts remain at docking station: PASS")
        print("Invisible unloading: NOT USED")
        print("Robot state unchanged: True")
        print("Panda preflight: PASS")
        print("AGV DIRECT PICK READY: True")
        print("Animation:", animation_path.resolve())

        # Encode the actual rendered AGV movement as MP4 for the dashboard.
        agv_movie = output_dir / f"agv_docking_{product_id}.mp4"
        with imageio.get_writer(str(agv_movie), fps=8, codec="libx264", macro_block_size=16) as writer:
            for frame in frames:
                writer.append_data(np.asarray(frame.convert("RGB")))
        print("AGV MP4:", agv_movie)



    except Exception:
        for color in COLORS:
            p.resetBasePositionAndOrientation(
                objects[color],
                object_positions[color],
                [0, 0, 0, 1]
            )

        for body, _ in visual_parts:
            p.removeBody(body)

        AGV_DIRECT_PICK_READY = False
        raise

    return {"agv_mp4":agv_movie.name,"agv_gif":animation_path.name,"docking_success":bool(AGV_DIRECT_PICK_READY)}


## 8｜既存3工程ロボット制御（結果収集のみ追加）

In [ ]:
def run_robot_sorting(record_stage):
    global cube, plan, results, placed_baselines, final_occupancy, expected_occupancy, all_success, state_preservation_log
    # =====================================================
    # STEP 10B: SOP-BASED THREE-PART SORTING
    # Uses validated plan freshly generated by Qwen (Step 11A)
    # Uses scene from Step 10A
    # =====================================================

    import numpy as np


    # Clean notebook: only the original motor-control functions are defined.


    # 1. Check required components
    required = [
        "complete_plan", "sop_lines", "verify_against_sop",
        "objects", "slot_positions", "object_positions",
        "observe_slot_occupancy", "set_light",
        "move_to", "hold_and_move_fingers",
        "ee_position", "cube_position", "finger_contacts"
    ]

    for name in required:
        assert name in globals(), f"Missing {name}"

    assert p.isConnected(), "PyBullet is disconnected"

    orientation = p.getQuaternionFromEuler([np.pi, 0, 0])

    def gate(condition, message):
        if not condition:
            raise RuntimeError("STOP: " + message)

    def part_position(name):
        return np.array(
            p.getBasePositionAndOrientation(objects[name])[0]
        )

    # 2. Verify planned tasks against SOP
    plan = complete_plan

    gate(len(plan) == len(sop_lines) == 3,
         "Unexpected number of tasks")

    for i, task in enumerate(plan):
        number, original_text = sop_lines[i]

        gate(
            task["step"] == int(number)
            and verify_against_sop(
                original_text,
                task["object_color"],
                task["slot"]
            ),
            f"SOP mismatch at step {i+1}"
        )

    gate(
        len({t["slot"] for t in plan}) == len(plan),
        "Duplicate destination slot"
    )

    # 3. Preflight: shared function used by fault-injection tests
    assert validate_preflight(plan)
    initial_occupancy = observe_slot_occupancy()

    for task in plan:
        color = task["object_color"]
        slot = task["slot"]

        gate(color in objects, f"Unknown part: {color}")
        gate(slot in slot_positions, f"Unknown slot: {slot}")

        pos = part_position(color)
        source = np.array(object_positions[color])

        gate(
            np.linalg.norm(pos - source) < 0.04,
            f"Missing part at source: {color}"
        )

        gate(
            len(initial_occupancy[slot]) == 0,
            f"Destination occupied: {slot}"
        )

    print("=== STEP 10B PREFLIGHT ===")
    print("Plan:", plan)
    print("Initial occupancy:", initial_occupancy)
    print("Preflight: PASS")

    # 4. Execute each task
    results = []
    placed_baselines = {}
    state_preservation_log = []

    for task in plan:

        step_id = task["step"]
        color = task["object_color"]
        slot = task["slot"]

        # Update active cube used by Step 6 functions
        cube = objects[color]

        print("\n" + "=" * 50)
        print(f"STEP {step_id}: {color} -> {slot}")
        print("=" * 50)

        # Check physical state again before each operation
        source_pos = part_position(color)
        source = np.array(object_positions[color])
        occupancy = observe_slot_occupancy()

        gate(
            np.linalg.norm(source_pos - source) < 0.04,
            f"Part missing before execution: {color}"
        )
        gate(
            len(occupancy[slot]) == 0,
            f"Slot occupied before execution: {slot}"
        )

        # Update visual light indicators
        for name in light_status:
            set_light(name, False)

        set_light(color, True)
        set_light(slot, True)

        print("Lights ON:", color, slot)

        # A. APPROACH
        print("[1/5] Approach")
        record_stage("APPROACH", step_id, color, slot)

        for z, steps, velocity in [
            (0.25, 1200, 0.5),
            (0.145, 1200, 0.25),
            (0.095, 960, 0.15),
            (float(source_pos[2]), 960, 0.12)
        ]:
            error = move_to(
                [source_pos[0], source_pos[1], z],
                steps=steps,
                velocity=velocity,
                gripper_open=True
            )

            gate(
                error < 0.03,
                f"Approach failed: {color}, Z={z}"
            )

            gate(
                np.linalg.norm(
                    part_position(color)[:2] - source_pos[:2]
                ) < 0.007,
                f"Part shifted during approach: {color}"
            )

        # B. GRASP
        print("[2/5] Grasp")
        record_stage("GRASP", step_id, color, slot)

        hold_and_move_fingers(False, steps=480)

        gate(
            {9, 10}.issubset(finger_contacts()),
            f"Grasp contact failed: {color}"
        )

        # C. LIFT
        print("[3/5] Lift")
        record_stage("LIFT", step_id, color, slot)

        cube_before = cube_position().copy()
        lift_target = ee_position().copy()
        lift_target[2] += 0.12

        error = move_to(
            lift_target,
            steps=1200,
            velocity=0.12,
            gripper_open=False
        )

        cube_lift = cube_position()[2] - cube_before[2]

        gate(
            error < 0.03
            and cube_lift > 0.07
            and {9, 10}.issubset(finger_contacts()),
            f"Lift failed: {color}"
        )

        # D. TRANSPORT
        print("[4/5] Transport")
        record_stage("TRANSPORT", step_id, color, slot)

        ee_before = ee_position().copy()
        cube_before = cube_position().copy()

        xy = np.array(slot_positions[slot])

        target = [
            float(xy[0]),
            float(xy[1]),
            float(ee_before[2])
        ]

        error = move_to(
            target,
            steps=1440,
            velocity=0.12,
            gripper_open=False
        )

        tracking_error = np.linalg.norm(
            (ee_position() - ee_before)
            - (cube_position() - cube_before)
        )

        gate(
            error < 0.03
            and tracking_error < 0.05
            and cube_position()[2] > 0.08
            and {9, 10}.issubset(finger_contacts()),
            f"Transport failed: {color}"
        )

        # E. PLACE AND RELEASE
        print("[5/5] Place and release")
        record_stage("PLACE_RELEASE", step_id, color, slot)

        ee_start = ee_position().copy()
        cube_start = cube_position().copy()

        place_target = ee_start.copy()
        place_target[2] -= (cube_start[2] - 0.045)

        for k in range(1, 11):
            waypoint = (
                ee_start
                + (place_target - ee_start) * (k / 10)
            )

            error = move_to(
                waypoint,
                steps=160,
                velocity=0.10,
                gripper_open=False
            )

            gate(
                error < 0.03,
                f"Place waypoint {k} failed: {color}"
            )
            gate(
                cube_position()[2] >= 0.03,
                f"Part dropped during placement: {color}"
            )

        hold_and_move_fingers(True, steps=480)


        # STEP 10F: Vertical retreat after releasing each part
        # First move upward, then proceed to the next task.

        retreat_xy = ee_position()[:2].copy()

        for retreat_z in [0.095, 0.145, 0.25]:

          retreat_target = [
            float(retreat_xy[0]),
            float(retreat_xy[1]),
            float(retreat_z)
          ]

          retreat_error = move_to(
            retreat_target,
            steps=720,
            velocity=0.10,
            gripper_open=True
          )

          print(
            f"Retreat Z={retreat_z:.3f}m, "
            f"error={retreat_error:.5f}m"
          )

          gate(
            retreat_error < 0.03,
            f"Retreat failed: {color}"
          )

        print("Vertical retreat completed")


        # Allow the released cube to settle
        for _ in range(120):
            p.stepSimulation()

        # F. VERIFY PHYSICAL RESULT
        final_pos = part_position(color)
        occupancy = observe_slot_occupancy()

        placement_error = np.linalg.norm(
            final_pos[:2] - xy
        )

        floor_contacts = p.getContactPoints(
            bodyA=cube, bodyB=plane
        )

        floor_ok = (
            0.020 <= final_pos[2] <= 0.040
            and len(floor_contacts) > 0
        )

        fingers_open = all(
            p.getJointState(robot, j)[0] > 0.035
            for j in [9, 10]
        )

        slot_ok = occupancy[slot] == [color]

        task_success = (
            floor_ok
            and fingers_open
            and slot_ok
            and placement_error < 0.03
        )

        print("Final position:", np.round(final_pos, 5))
        print("Placement error (m):",
              round(float(placement_error), 5))
        print("Tracking error (m):",
              round(float(tracking_error), 5))
        print("Slot occupants:", occupancy[slot])
        print("Cube on floor:", floor_ok)
        print("Fingers open:", fingers_open)
        print("Task success:", task_success)

        gate(task_success, f"Final validation failed: {color}")

        # State Preservation Gate
        # Verify that previously placed objects remain in place.

        for previous_color, baseline in placed_baselines.items():
            current = part_position(previous_color)

            shift = np.linalg.norm(current - baseline)

            print(
                f"State preservation [{previous_color}]:",
                f"{shift * 1000:.2f} mm"
            )
            state_preservation_log.append({
                "after_step": int(step_id),
                "previous_color": previous_color,
                "displacement_mm": float(shift * 1000)
            })

            gate(
                shift < 0.01,
                f"Previously placed part moved: {previous_color}"
            )

        # Register successfully placed object's final position
        placed_baselines[color] = part_position(color).copy()

        results.append({
            "step": step_id,
            "color": color,
            "slot": slot,
            "placement_error_m": float(placement_error),
            "tracking_error_m": float(tracking_error),
            "success": task_success
        })

        # Turn off lights after completion
        set_light(color, False)
        set_light(slot, False)

    # 5. Final summary
    print("\n" + "=" * 55)
    print("STEP 10B - FINAL SORTING RESULT")
    print("=" * 55)

    for result in results:
        print(
            f"Step {result['step']}: "
            f"{result['color']} -> {result['slot']} | "
            f"Success={result['success']} | "
            f"Placement error="
            f"{result['placement_error_m']*1000:.2f}mm"
        )

    final_occupancy = observe_slot_occupancy()

    print("\nFinal occupancy:", final_occupancy)


    # Expected final state is derived from the validated plan
    expected_occupancy = {
        slot: [] for slot in slot_positions
    }

    for task in plan:
        expected_occupancy[task["slot"]].append(
            task["object_color"]
        )

    all_success = (
        len(results) == len(plan)
        and all(bool(r["success"]) for r in results)
        and final_occupancy == expected_occupancy
    )

    print("Expected occupancy:", expected_occupancy)


    print("SOP SORTING SUCCESS:", all_success)
    assert all_success, "STOP: Final occupancy or motion validation failed"

    # Show final scene
    w, h, rgba, _, _ = p.getCameraImage(
        960, 640,
        viewMatrix=view,
        projectionMatrix=projection,
        renderer=p.ER_TINY_RENDERER
    )

    display(
        Image.fromarray(
            np.asarray(rgba, dtype=np.uint8).reshape(h, w, 4)
        ).convert("RGB")
    )

    return {
        "steps": results,
        "final_occupancy":final_occupancy,
        "expected_occupancy":expected_occupancy,
        "all_success": bool(all_success),
        "state_preservation":state_preservation_log
    }


## 9｜MP4記録、異常ゲート、JSONシリアライズ

In [ ]:
import imageio.v2 as imageio
from datetime import datetime, timezone
from pathlib import Path
import json, platform, sys
from importlib.metadata import version, PackageNotFoundError

def json_safe(v):
    if isinstance(v,np.ndarray): return v.tolist()
    if isinstance(v,np.generic): return v.item()
    if isinstance(v,Path): return str(v)
    raise TypeError(f"not JSON serializable: {type(v).__name__}")

def pkg(name):
    try: return version(name)
    except PackageNotFoundError: return "unknown"

class RobotRecorder:
    """Passively captures rendered frames; wraps, but does not change, simulation stepping."""
    def __init__(self, movie_path):
        self.movie_path=movie_path
        self.frame_count=0
        self.physics_steps=0
        self.stages=[]
        self.original_step=None
        self.writer=None

    def capture(self):
        w,h,rgba,_,_=p.getCameraImage(
            VIDEO_W, VIDEO_H, viewMatrix=view, projectionMatrix=projection,
            renderer=p.ER_TINY_RENDERER
        )
        img=np.asarray(rgba,dtype=np.uint8).reshape(h,w,4)[:,:,:3]
        self.writer.append_data(img)
        self.frame_count+=1

    def stage(self,stage,step,color,slot):
        self.stages.append({
            "stage": stage,"step":int(step),"color":color,"slot":slot,
            "video_time_s":float(self.frame_count/VIDEO_FPS),
            "frame_index":int(self.frame_count)
        })

    def __enter__(self):
        self.writer=imageio.get_writer(
            str(self.movie_path),fps=VIDEO_FPS,codec="libx264",macro_block_size=16
        )
        self.original_step=p.stepSimulation
        self.capture()
        def traced_step(*args,**kwargs):
            self.original_step(*args,**kwargs)
            self.physics_steps+=1
            if self.physics_steps % VIDEO_CAPTURE_EVERY_STEPS == 0:
                self.capture()
        p.stepSimulation=traced_step
        return self

    def __exit__(self,exc_type,exc_value,traceback):
        if self.original_step is not None:
            p.stepSimulation=self.original_step
        try:
            if self.writer is not None:
                self.capture()
        finally:
            if self.writer is not None:
                self.writer.close()

def run_video_and_robot(product_id, outdir):
    video_file=outdir/f"robot_sorting_{product_id}.mp4"
    with RobotRecorder(video_file) as rec:
        def record_stage(stage,step,color,slot):
            rec.stage(stage,step,color,slot)
        execution=run_robot_sorting(record_stage)
    execution["robot_video"]=video_file.name
    execution["timeline_events"]=rec.stages
    execution["video_frames"]=rec.frame_count
    execution["video_fps"]=VIDEO_FPS
    execution["recorded_physics_steps"]=rec.physics_steps
    return execution


def run_safety_smoke_tests(plan):
    """Same logical preflight routine as execution; no robot motion."""
    assert validate_preflight(plan)
    def fault(color,location,reason):
        body=objects[color]
        oldpos,oldorn=p.getBasePositionAndOrientation(body)
        oldlin,oldang=p.getBaseVelocity(body)
        js=[p.getJointState(robot,j)[0] for j in list(range(7))+[9,10]]
        blocked=False
        try:
            p.resetBasePositionAndOrientation(body,location,[0,0,0,1])
            p.resetBaseVelocity(body,[0,0,0],[0,0,0])
            try: validate_preflight(plan)
            except RuntimeError as e: blocked=(str(e)==reason)
        finally:
            p.resetBasePositionAndOrientation(body,oldpos,oldorn)
            p.resetBaseVelocity(body,oldlin,oldang)
        assert blocked and validate_preflight(plan)
        assert np.allclose(js,[p.getJointState(robot,j)[0] for j in list(range(7))+[9,10]],atol=1e-9)
        return {"expected_error":reason,"correctly_blocked":bool(blocked),"robot_unchanged":True}
    res=[fault("red",[0.80,0.16,0.025],"Missing part at source: red"),
         fault("green",[float(slot_positions["A1"][0]),float(slot_positions["A1"][1]),0.025],"Occupied slot: A1")]
    print("FAULT INJECTION: PASS (missing part / occupied slot)")
    return res


## 10｜A/Bを上から一括実行し、Driveへ保存

In [ ]:
# ================================================================
# MAIN: New runtime -> Run all -> A and B, unique logs and MP4s
# ================================================================
from datetime import datetime, timezone
from pathlib import Path

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    base_dir=Path("/content/drive/MyDrive/PAI2026_Submission_Artifacts")
else:
    base_dir=Path("/content/PAI2026_Submission_Artifacts")

run_id=datetime.now(timezone.utc).strftime("run_%Y%m%dT%H%M%SZ")
run_dir=base_dir/run_id
run_dir.mkdir(parents=True, exist_ok=False)
print("Run output folder:",run_dir)

all_entries=[]
for product_id in PRODUCT_SEQUENCE:
    print("\n"+"="*70)
    print("START PRODUCT:",product_id)
    print("="*70)
    outdir=run_dir/product_id
    outdir.mkdir()

    # A fresh simulation per product. No stale cubes or robot states.
    initialize_scene()
    agv_data=dock_agv(product_id,outdir)
    qr_data=scan_qr_and_plan(product_id,outdir)  # real Qwen inference
    assert qr_data["product_id"]==product_id
    SOP_TEXT=qr_data["sop"]
    sop_lines=re.findall(r"^工程\s*(\d+)\s*[：:]\s*(.+)$",SOP_TEXT,flags=re.MULTILINE)
    complete_plan=[dict(x) for x in qr_data["plan"]]
    candidate_plan=[dict(x) for x in complete_plan]
    step11_raw_results=qr_data["raw_llm_outputs"]
    assert validate_preflight(complete_plan)
    safety_checks=run_safety_smoke_tests(complete_plan)

    if not CAPTURE_ROBOT_VIDEO:
        raise RuntimeError("The submission capture notebook requires real robot video")
    robot_data=run_video_and_robot(product_id,outdir)
    assert robot_data["all_success"], "Robot task failed"

    manifest={
        "run_id":run_id,"product_id":product_id,
        "work_order_id":qr_data["work_order_id"],
        "recorded_at_utc":datetime.now(timezone.utc).isoformat(),
        "inference":{"model_id":MODEL_ID,"inference_api_used":False,
                     "runtime":"Google Colab cloud", "duration_s":qr_data["llm_inference_seconds"],
                     "raw_step_outputs":step11_raw_results},
        "sop":{"text":SOP_TEXT,"validated_plan":complete_plan},
        "qr":{"decoded_id":qr_data["work_order_id"],"image_file":qr_data["qr_image"],
              "source":"generated PNG decoded by OpenCV, not a PyBullet virtual camera"},
        "agv":{"video":agv_data["agv_mp4"],"gif":agv_data["agv_gif"],
               "mode":"kinematic synchronized transport; visual support; no SLAM or wheel dynamics"},
        "validation":{"preflight_pass":True,"fault_tests":safety_checks},
        "robot_execution":robot_data,
        "environment":{"python":sys.version.split()[0],"platform":platform.platform(),
            "device":str(device),"versions":{k:pkg(k) for k in ["pybullet","numpy","torch","transformers"]}}
    }
    path=outdir/f"manifest_{product_id}.json"
    path.write_text(json.dumps(manifest,ensure_ascii=False,indent=2,default=json_safe),encoding="utf-8")
    saved=json.loads(path.read_text(encoding="utf-8"))
    assert saved["robot_execution"]["all_success"] and len(saved["robot_execution"]["steps"])==3
    all_entries.append({
        "product_id":product_id,"work_order_id":qr_data["work_order_id"],
        "manifest":str(path.relative_to(run_dir)),
        "agv_video":str((outdir/agv_data["agv_mp4"]).relative_to(run_dir)),
        "robot_video":str((outdir/robot_data["robot_video"]).relative_to(run_dir)),
        "qr_image":str((outdir/qr_data["qr_image"]).relative_to(run_dir)),
        "success":True
    })
    print("PRODUCT",product_id,"RUN/PERSISTENCE: PASS")

assert len(all_entries)==len(PRODUCT_SEQUENCE)
if len(PRODUCT_SEQUENCE)==2:
    assert all_entries[0]["product_id"]!=all_entries[1]["product_id"]
(run_dir/"dashboard_index.json").write_text(json.dumps({
    "run_id":run_id,"replay_only":True,
    "description":"Recorded Colab/PyBullet outputs, not browser-side live inference/control",
    "products":all_entries
},ensure_ascii=False,indent=2),encoding="utf-8")
print("\n"+"="*60)
print("SUBMISSION CAPTURE RESULT")
print("="*60)
for item in all_entries:
    print(item["product_id"],"QR / QWEN / AGV / ROBOT / JSON: PASS")
print("ALL PRODUCT CAPTURES SUCCESS: True")
print("ARTIFACTS:",run_dir)


## 11｜実行結果・レポート・再現性

**検証済みの記録**：`run_20261029T124105Z`。製品A/Bの各1回の実行で、QR読取・Qwen推論・Validator・AGV模擬搬送・ロボット3工程・JSON/MP4保存が完了しました。記録値は `assets/PRODUCT_A/manifest_PRODUCT_A.json` と `assets/PRODUCT_B/manifest_PRODUCT_B.json` に保存済みです。

- 製品A: 赤→A1、青→B1、緑→A2。3/3工程成功。最大配置誤差14.24 mm、最大状態変位1.20 mm。
- 製品B: 赤→A1、青→A2、緑→B1。3/3工程成功。最大配置誤差14.24 mm、最大状態変位1.20 mm。
- `dashboard_index.json` と実行JSONの相対パスで動画を対応づけています。`timeline_events` は動画内部の時刻に対応し、Colab推論中の時刻とのリアルタイム同期ではありません。
- 同じ3色各1個の配置先を切り替えた実験です。**品種変更による部品数量の変更（BOMの変更）は検証していません**。
- 入力JSONの書式や欠品・占有の検査はPythonで実施。実機の機能安全や未知指示への汎化は検証していません。

**再現性について**：上記は記録済み実験の結果であり、再実行時に同一結果を保証するものではありません。GPU環境・パッケージ更新・シミュレーションの非決定性によって値が変化し得ます。
